# Synthetic ALM hedging research

This executable narrative values representative indexed liabilities, measures risk, runs full-revaluation stresses, and constructs an illustrative hedge. It calls the production `alm_hedging` package throughout. Inputs and results are synthetic; this is not investment advice or a production insurance model.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import alm_hedging as ah

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
assumptions = json.loads((ROOT / "data/assumptions_v1.json").read_text())
cohort_data = pd.read_csv(ROOT / "data/cohorts_v1.csv")
assumptions["model_version"], cohort_data[["cohort_id", "product", "policy_count"]]

## Market and representative liabilities

The compact example uses the deterministic synthetic market builder and one cohort for each modeled product family. Rates, dividends, and volatilities are decimal values.

In [ ]:
market = ah.SyntheticMarketModel(base_rate=0.03, dividend_yield=0.015, atm_vol=0.20).build()
liabilities = [
    ah.Cohort(ah.RILABufferedSegment(100_000, 1.0, buffer=0.10, cap=0.18, floor=-0.50), 2500, label="RILA-2025-A"),
    ah.Cohort(ah.FIAAnnualPointToPoint(75_000, 1.0, annual_cap=0.095, participation=0.85), 4000, label="FIA-2025-A"),
    ah.Cohort(ah.IULAccountValue(85_000, 5.0, annual_cap=0.11, annual_policy_charge=0.0125, mortality_rates=(0.003,), lapse_rates=(0.035,)), 1800, label="IUL-2025-A"),
]
values = [ah.portfolio_value([item], market, n_paths=20_000, seed=20250102) for item in liabilities]
pd.DataFrame({"cohort": [item.label for item in liabilities], "pv": values})

## Risk and full-revaluation stresses

For an interactive notebook runtime, Greeks below use the two analytically valued one-year cohorts. The production API applies central bumps with the documented conventions.

In [ ]:
analytic_book = liabilities[:2]
pricer = lambda state: ah.portfolio_value(analytic_book, state)
greeks = ah.bump_and_revalue(pricer, market)
pd.Series(greeks.as_dict(), name="liability_risk")

In [ ]:
stress_results = ah.run_stress_scenarios(
    market,
    ah.standard_stress_scenarios(),
    lambda state, book: ah.portfolio_value(book, state),
    analytic_book,
)
pd.DataFrame([{"scenario": x.scenario, "base_pv": x.base_value, "stressed_pv": x.stressed_value, "pnl": x.pnl} for x in stress_results])

## Illustrative constrained hedge

The optimizer adds instrument exposure to the unhedged liability exposure, so recommended positions should oppose the liability risk. Unit exposures below are deliberately stylized rather than dealer quotes.

In [ ]:
instruments = [
    ah.HedgeInstrument("index_future", "future", 1.0, 100.0, {"delta": 1.0, "vega": 0.0}, max_position=2e8, max_trade=2e8, lot_size=1.0),
    ah.HedgeInstrument("atm_option", "option", 1.0, 8.0, {"delta": 0.5, "vega": 35.0}, premium=8.0, max_position=2e7, max_trade=2e7, lot_size=1.0),
]
solution = ah.optimize_hedge(
    {"delta": greeks.delta, "vega": greeks.vega},
    instruments,
    exposure_weights={"delta": 1.0, "vega": 0.01},
    constraints=ah.HedgeConstraints(premium_budget=1e9, turnover_budget=1e11),
)
pd.DataFrame({"position": solution.positions, "trade": solution.trades})

The recommendation reduces modeled delta and vega subject to simplified implementation limits. Residual risk includes nonlinear payoff, basis, behavior, liquidity, and model risk. See `docs/exclusions.md` before interpreting the result.